# Recommendation baseline comparison

Comparison of non-personalized public-quality ranking and content-based personalization. The experiment evaluates whether a preference profile improves retrieval of highly rated dramas over a simple public-rating baseline.

| Component | Role |
|---|---|
| Enriched watch history | Supplies metadata and observed personal feedback |
| Separate candidate catalog | Supplies titles absent from recorded history |
| Ranking functions | Return ordered candidates with explicit score types |
| Held-out evaluation | Measures recovery of known preferences |

## Inputs and configuration

Install with `python -m pip install -e ".[notebook]"`. Prepare `data/processed/watch_history_enriched.csv` and `data/processed/candidate_catalog.csv` using the repository workflows. These local snapshots are not bundled with the public notebooks.

Copy this notebook to `notebooks/local/` before execution. Published notebooks contain no personal outputs. No network calls, agent framework or probability prediction model is involved.

Configuration is fixed before evaluation: relevance threshold 8/10, five stratified folds, seed 42, cutoff five, prior strength 1,000 votes, and an exploratory cast weight of 0.2. Reusing evaluation results to choose new settings would require a separate final assessment.


In [ ]:
from pathlib import Path
import os
import hashlib
import json
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
os.environ.setdefault('MPLCONFIGDIR', str(root / 'data/processed/.matplotlib'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from dr_drama.analysis.eda import load_history
from dr_drama.models.baseline import ContentProfile, rank_public, public_prior
from dr_drama.models.evaluation import evaluate_baselines

history_path = root / 'data/processed/watch_history_enriched.csv'
catalog_path = root / 'data/processed/candidate_catalog.csv'
for path in [history_path, catalog_path]:
    if not path.exists():
        raise FileNotFoundError(f"Required local snapshot missing: {path.name}. See the repository README.")
history_hash = hashlib.sha256(history_path.read_bytes()).hexdigest()
catalog_hash = hashlib.sha256(catalog_path.read_bytes()).hexdigest()
history = load_history(history_path)
catalog = pd.read_csv(catalog_path, dtype=str, keep_default_na=False)
THRESHOLD, FOLDS, SEED, K = 8, 5, 42, 5
CAST_WEIGHT, PRIOR_VOTES = .2, 1000
plt.rcParams.update({'figure.figsize': (8, 4.5), 'axes.spines.top': False, 'axes.spines.right': False})
pd.set_option('display.max_columns', 20)
display(history.status.value_counts().rename('dramas').to_frame())
display(history[['genres', 'main_cast', 'release_year', 'number_of_episodes', 'public_rating', 'public_vote_count']].isna().sum().rename('missing').to_frame())
if 'mdl_match_status' in history:
    display(history.mdl_match_status.value_counts().rename('identity_review').to_frame())
display(Markdown(f"**{len(history)} records**, {history.external_api_id.nunique()} unique IDs; **{((history.status == 'completed') & (history.my_rating >= THRESHOLD)).sum()} completed 8+ dramas**."))


## Ranking methods

**A — Public quality.** Rank by `(votes × public_rating + prior_votes × prior) / (votes + prior_votes)`. The prior is the training-item mean public rating; unsupported ratings receive the prior with an explicit flag. Ties use ascending popularity rank and then ID. The history-derived prior reflects a selected sample, not the full audience population.

**B — Genre similarity.** Form a normalized centroid of unit genre vectors from completed training dramas rated 8+. Rank candidates by cosine similarity to that profile. Genre memberships are independent indicators; duplicate labels do not increase weight.

**B with cast.** Combine 0.8 genre similarity with 0.2 main-cast similarity as a fixed feature comparison. Sparse credits and name aliases limit cast coverage.

All recorded history IDs are excluded from live suggestions, including dropped and unfinished shows. Missing or unknown attributes provide no overlap. Without a usable positive profile, B explicitly falls back to A. Public-quality and similarity scores have different scales; neither is a calibrated enjoyment probability.

## Feature scope

| Feature | Use |
|---|---|
| Genres | Primary content representation |
| Main cast | Exploratory content variant |
| Public rating and vote count | Public-quality baseline |
| Popularity | Tie-breaker for public ranking |
| Release year and episode count | Descriptive output only; not similarity inputs |
| Notes, favorite and rewatch flags | Excluded from scoring to prevent outcome leakage |

## Candidate rankings

The following tables show the top candidates for each method, including shared attributes for content rankings. Catalog coverage directly limits discovery quality. A catalog derived from alternate title-search results is provisional and search-biased; identity, series prerequisites and availability require review before relying on suggestions.


In [ ]:
display(catalog[['canonical_title', 'release_year', 'number_of_episodes', 'public_rating', 'public_vote_count', 'catalog_status', 'metadata_retrieved_at']])
profile = ContentProfile.fit(history, threshold=THRESHOLD)
cast_profile = ContentProfile.fit(history, threshold=THRESHOLD, cast_weight=CAST_WEIGHT)
rankings = {
    'A_public': rank_public(catalog, history, prior_votes=PRIOR_VOTES),
    'B_genres': profile.rank(catalog, history),
    'B_genres_cast': cast_profile.rank(catalog, history),
}
for name, ranking in rankings.items():
    assert not set(ranking.external_api_id) & set(history.external_api_id)
    display(Markdown(f'### {name}: top {K} provisional suggestions'))
    columns = ['canonical_title', 'release_year', 'number_of_episodes', 'score', 'score_kind', 'source_url']
    if name != 'A_public':
        columns += ['shared_genres']
    if name == 'B_genres_cast':
        columns += ['shared_main_cast']
    display(ranking[columns].head(K).round(4))
display(Markdown(f'Public prior: **{public_prior(history):.3f}**. Profile uses **{len(profile.positive_ids)}** completed 8+ dramas. Scores across A and B have different meanings and should not be compared numerically.'))


## Retrospective evaluation

Completed, rated dramas are divided into stratified folds by 8+ relevance. Each drama appears in one held-out candidate pool. Profiles, feature vocabularies and public priors are fitted using only the other folds; candidate feedback is removed before ranking. Both methods rank identical candidates.

| Metric | Interpretation |
|---|---|
| Precision@5 | Proportion of the first five candidates rated 8+ |
| Recall@5 | Proportion of available held-out 8+ dramas recovered in the first five |
| NDCG@5 | Rank-sensitive relevance relative to an ideal ordering |

The random expected precision is the positive fraction of each candidate pool. Fold means and ranges summarize this development comparison; overlapping training sets mean fold spread is not a confidence interval. Without viewing dates or historical public metrics, these results do not represent a chronological backtest.


In [ ]:
metrics, predictions = evaluate_baselines(history, folds=FOLDS, seed=SEED, k=K, threshold=THRESHOLD,
                                               cast_weight=CAST_WEIGHT, prior_votes=PRIOR_VOTES)
metric_columns = ['precision_at_k', 'recall_at_k', 'ndcg_at_k']
summary = metrics.groupby('model')[metric_columns].mean()
display(metrics.round(4))
display(summary.round(4))
display(metrics.groupby('model')[metric_columns].agg(['min', 'max']).round(4))
random_precision = metrics.drop_duplicates('fold').random_expected_precision.mean()
fig, ax = plt.subplots()
summary.plot.bar(ax=ax, rot=0, color=['#4477AA', '#66AABB', '#AA6677'])
ax.set(ylim=(0,1), ylabel='Mean held-out ranking metric', xlabel='', title='Five-fold retrospective comparison (k=5)')
ax.legend(['Precision@5', 'Recall@5', 'NDCG@5'], loc='upper right')
fig.tight_layout(); display(fig); plt.close(fig)
a, b, bc = (summary.loc[name] for name in ['A_public', 'B_genres', 'B_genres_cast'])
conclusion = ('Public ranking outperformed both personalized variants on mean Precision@5.'
              if a.precision_at_k > max(b.precision_at_k, bc.precision_at_k)
              else 'Inspect paired folds and all metrics before claiming a personalization benefit.')
display(Markdown(f"**RESULT:** A Precision@5 = **{a.precision_at_k:.2f}**; genre B = **{b.precision_at_k:.2f}**; genre+cast B = **{bc.precision_at_k:.2f}**. Random expected precision = **{random_precision:.3f}**. {conclusion}"))
display(Markdown('These figures measure recovery of known preferences from a selected history, not your probability of liking a new drama. Adding cast is exploratory; it does not justify choosing a production model from these same folds.'))


## Interpretation and operating limits

Compare all metrics and folds before concluding that personalization adds value. A positive-only genre profile can overemphasize attributes common to both liked and lower-rated dramas. Adding cast may change ordering without improving relevant-item counts.

The experiment is restricted to completed titles from a selected history. Unwatched titles are unlabeled, not negative outcomes. Small samples, provisional metadata, partial cast, franchise overlap and current public ratings limit generalization. Recommendations do not enforce streaming availability, sequel dependencies or diversity constraints.

Enjoyment probabilities require a separately defined outcome, sufficient representative feedback, a supervised model and independent calibration assessment. A similarity score in the range 0–1 does not imply probability calibration.

## Snapshot integrity

The following check verifies that analysis has not modified either input. Results remain in the executed local notebook; this public version does not export or embed personal findings.


In [ ]:
assert hashlib.sha256(history_path.read_bytes()).hexdigest() == history_hash
assert hashlib.sha256(catalog_path.read_bytes()).hexdigest() == catalog_hash
print('Input snapshots unchanged.')